# Example 16.9 / 1: Solve a Small Polygon With a Local Solver

Companion notebook for **16.9 Largest Small Polygon** in the AMPL lab Documentation, section "Solve a Small Polygon With a Local Solver". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a Small Polygon With a Local Solver

The model below is the polar formulation with the cyclic area objective. Because `radius[1] = 0`, every shoelace term that contains vertex 1 vanishes, and the area is $\frac12\sum_{i<n} r_i r_{i+1}\sin(\theta_{i+1}-\theta_i)$. The initial values follow the source notebook. The run solves with the local solver Ipopt, reads the objective with `ampl.obj[...].value()`, and recomputes the largest squared distance with `ampl.get_value`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n integer >= 3;  # number of polygon vertices
param PI := 4 * atan(1);  # pi, for the angle bounds
set V ordered := 1..n;  # vertices in angular order
var radius {i in V} >= 0, <= 1 := 4*i*(n+1-i)/(n+1)^2;  # polar radius; start from the source
var angle {i in V} >= 0, <= 2*PI := PI*i/n;  # polar angle, radians; evenly spaced start

subject to AnchorRadius:  # first vertex at the origin removes translation
    radius[first(V)] = 0;
subject to AnchorAngle:  # first angle 0 removes rotation
    angle[first(V)] = 0;
subject to AngleOrder {i in V: ord(i) > 1}:  # vertices ordered by angle
    angle[i] >= angle[prev(i)];
subject to UnitDiameter {i in V, j in V: ord(i) < ord(j)}:  # every pair at most 1 apart
    radius[i]^2 + radius[j]^2
    - 2*radius[i]*radius[j]*cos(angle[i]-angle[j]) <= 1;

maximize Area:  # polygon area by the polar shoelace formula
    0.5 * sum {i in V: ord(i) < n}
        radius[i]*radius[next(i)]*sin(angle[next(i)]-angle[i]);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["n"] = 5

In [5]:
# run.py
for size in [5, 6]:
    ampl.param["n"] = size
    ampl.solve(solver="ipopt", verbose=False)
    diameter_sq = ampl.get_value(
        "max {i in V, j in V: ord(i) < ord(j)} (radius[i]^2 + radius[j]^2"
        " - 2*radius[i]*radius[j]*cos(angle[i]-angle[j]))"
    )
    print(size, ampl.solve_result, round(ampl.obj["Area"].value(), 6), round(diameter_sq, 6))

5 solved 0.657164 1.0
6 solved 0.674981 1.0


### Another way in amplpy

For $n=5$ the local solve matches the regular pentagon bound 0.657164. For $n=6$ it returns 0.674981, above the regular hexagon value 0.649519, which shows concretely that the regular $n$-gon is not optimal for $n=6$. The largest squared distance is 1 within the printed rounding, so the diameter constraint is tight. The run below, with the same model.mod and data.py, solves $n=6$ from eight random starts: `Uniform(0,1)` radii and `Uniform(0, 2*PI)` angles for vertices 2 to 6 with `randseed` 1 to 8, drawn with `ampl.eval` because AMPL's `Uniform` function has no amplpy counterpart:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
ampl.param["n"] = 5

In [8]:
# run.py
ampl.param["n"] = 6
for seed in range(1, 9):
    ampl.option["randseed"] = seed
    ampl.eval("let {i in V: ord(i) > 1} radius[i] := Uniform(0, 1);")
    ampl.eval("let {i in V: ord(i) > 1} angle[i] := Uniform(0, 2*PI);")
    ampl.solve(solver="ipopt", verbose=False)
    diameter_sq = ampl.get_value(
        "max {i in V, j in V: ord(i) < ord(j)} (radius[i]^2 + radius[j]^2"
        " - 2*radius[i]*radius[j]*cos(angle[i]-angle[j]))"
    )
    print(seed, ampl.solve_result, round(ampl.obj["Area"].value(), 6), round(diameter_sq, 6))

1 solved 0.674981 1.0
2 solved 0.674981 1.0
3 solved 0.674981 1.0
4 solved 0.674981 1.0
5 solved 0.674981 1.0
6 solved 0.674981 1.0
7 solved 0.674981 1.0
8 solved 0.674981 1.0


### Pure AMPL: data.dat

All eight starts return 0.674981 with largest squared distance 1. That agreement is computational evidence; Ipopt's `solved` status still means a locally best polygon, and only a global solver that closes a valid upper bound proves that no larger area exists.

> **Pitfall: do not give the anchored vertex a random start**
> In a test run that also drew `radius[1]` and `angle[1]` at random, every solve after the first still reported `solved`, but with `angle[1]` left at its random value: the anchors were not met, and the returned "polygons" had largest distances up to 1.86 and areas up to 0.815469, above the true optimum. Only the recomputed diameter exposed the error.

For pure AMPL, the data in AMPL data format; with this data.dat, the first run above prints the same output. In pure AMPL, `option solver ipopt;`, `solve;` and `display Area;` do the same work, and Ipopt then also prints its banner and log:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
%%writefile data.dat
param n := 5;

Writing data.dat


In [11]:
%%ampl_eval
data data.dat;

In [12]:
# run.py
for size in [5, 6]:
    ampl.param["n"] = size
    ampl.solve(solver="ipopt", verbose=False)
    diameter_sq = ampl.get_value(
        "max {i in V, j in V: ord(i) < ord(j)} (radius[i]^2 + radius[j]^2"
        " - 2*radius[i]*radius[j]*cos(angle[i]-angle[j]))"
    )
    print(size, ampl.solve_result, round(ampl.obj["Area"].value(), 6), round(diameter_sq, 6))

5 solved 0.657164 1.0
6 solved 0.674981 1.0
